# Programación orientada a objetos con Python
## Video 8. Composición frente a herencia y proyecto integrador

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Distinguir con rigor entre una relación de especialización («es un», herencia) y una relación de agregación («tiene un», composición).
2. Evitar jerarquías de herencia artificiales modelando figuras geométricas y estructuras algebraicas mediante composición.
3. Construir el proyecto integrador de la serie: polinomios con coeficientes racionales en $\mathbb{Q}[x]$ y funciones racionales en $\mathbb{Q}(x)$ sobre `Fraccion`.
4. Calcular evaluaciones, derivadas e integrales definidas con aritmética racional exacta, sin errores de redondeo en punto flotante.

## 1. Planteamiento del problema

**Definición.**
1. Un *polinomio sobre los números racionales*, $P(x) \in \mathbb{Q}[x]$, es una expresión de la forma
   $$P(x) = \sum_{k=0}^{n} c_k\,x^k, \qquad c_k \in \mathbb{Q}.$$
2. Una *función racional*, $R(x) \in \mathbb{Q}(x)$, es un cociente formal
   $$R(x) = \frac{P(x)}{Q(x)},$$
   donde $P(x), Q(x) \in \mathbb{Q}[x]$ y $Q(x)$ no es el polinomio nulo.

**Problema: integración de toda la serie mediante composición y aritmética exacta.**

- **Entrada:** nuestra clase encapsulada `Fraccion` (que representa elementos exactos de $\mathbb{Q}$).
- **Salida:** las clases `PolinomioRacional` (compuesta por una tupla de instancias de `Fraccion`) y `FuncionRacional` (compuesta por dos instancias de `PolinomioRacional`), capaces de sumar, multiplicar, derivar, integrar exactamente en $[a, b]$ y evaluar sin pérdida de precisión.

## 2. Composición frente a herencia en objetos matemáticos

Antes de escribir `class Hijo(Padre):`, conviene formular una prueba conceptual estricta: **¿un `Hijo` *es un* caso particular de `Padre`, o *tiene un* `Padre` como componente?**

- Una `Circunferencia` en el plano **no es** un `VectorPlano`; **tiene** un centro (un `VectorPlano`) y un radio positivo. Si heredara de `VectorPlano`, expondría operaciones sin sentido geométrico para una circunferencia, como su norma o su producto interior.
- Una `FuncionRacional` **no es** un `PolinomioRacional`; **tiene** un polinomio numerador y un polinomio denominador.

Modelar mediante **composición** mantiene las interfaces limpias, desacopla la implementación interna y permite combinar libremente los objetos ya construidos en videos anteriores.

In [ ]:
from math import gcd, pi, sqrt


class VectorPlano:
    def __init__(self, x, y):
        self._x = float(x)
        self._y = float(y)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __add__(self, otro):
        if not isinstance(otro, VectorPlano):
            return NotImplemented
        return VectorPlano(self._x + otro._x, self._y + otro._y)

    def distancia(self, otro):
        return sqrt((self._x - otro._x) ** 2 + (self._y - otro._y) ** 2)

    def __repr__(self):
        return f"VectorPlano({self._x}, {self._y})"


class Circunferencia:
    def __init__(self, centro, radio):
        if not isinstance(centro, VectorPlano):
            raise TypeError("El centro debe ser una instancia de VectorPlano.")
        radio = float(radio)
        if radio <= 0:
            raise ValueError("El radio debe ser estrictamente positivo.")
        self._centro = centro
        self._radio = radio

    @property
    def centro(self):
        return self._centro

    @property
    def radio(self):
        return self._radio

    @property
    def area(self):
        return pi * self._radio ** 2

    def contiene_punto(self, punto):
        return self._centro.distancia(punto) <= self._radio

    def trasladar(self, desplazamiento):
        return Circunferencia(self._centro + desplazamiento, self._radio)

    def __repr__(self):
        return f"Circunferencia({self._centro!r}, {self._radio})"


circulo = Circunferencia(VectorPlano(1, 2), 5)
trasladado = circulo.trasladar(VectorPlano(3, -2))
print("Circunferencia:", circulo, "| área:", round(circulo.area, 4))
print("¿Contiene al punto (4, 6)?", circulo.contiene_punto(VectorPlano(4, 6)))
print("Trasladada:", trasladado)

### Tabla comparativa: herencia frente a composición

| Criterio | Herencia («es un») | Composición («tiene un») |
|---|---|---|
| Relación conceptual | Especialización de un tipo base | Agregación de partes independientes |
| Acoplamiento | Fuerte: expone toda la interfaz del padre | Débil: expone solo los métodos propios del contenedor |
| Reutilización | Automática mediante delegación de métodos | Explícita invocando operaciones sobre los componentes |
| Ejemplos en la serie | `PolinomioCuadratico(Polinomio)`, `EnteroRacional(Fraccion)` | `Circunferencia` con `VectorPlano`, `FuncionRacional` con `PolinomioRacional` |

## 3. Proyecto integrador — Parte 1: la clase `Fraccion` completa

Reunimos todas las técnicas de la serie en la versión final de `Fraccion`: encapsulamiento con `_numerador` y `_denominador` inmutables, `@property`, método estático `maximo_comun_divisor`, constructores `@classmethod` (`convertir`, `cero`, `uno`) y los métodos especiales de representación, comparación y aritmética completa.

In [ ]:
class Fraccion:
    def __init__(self, numerador, denominador=1):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = self.maximo_comun_divisor(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @staticmethod
    def maximo_comun_divisor(a, b):
        return gcd(a, b)

    @classmethod
    def cero(cls):
        return cls(0, 1)

    @classmethod
    def uno(cls):
        return cls(1, 1)

    @classmethod
    def convertir(cls, valor):
        if isinstance(valor, cls):
            return valor
        if isinstance(valor, int) and not isinstance(valor, bool):
            return cls(valor, 1)
        raise TypeError(f"No se puede convertir {valor!r} en una Fraccion.")

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    def __repr__(self):
        return f"Fraccion({self._numerador}, {self._denominador})"

    def __str__(self):
        if self._denominador == 1:
            return str(self._numerador)
        return f"{self._numerador}/{self._denominador}"

    def __eq__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return self._numerador == otra_f._numerador and self._denominador == otra_f._denominador

    def __lt__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return self._numerador * otra_f._denominador < self._denominador * otra_f._numerador

    def __neg__(self):
        return Fraccion(-self._numerador, self._denominador)

    def __add__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return Fraccion(
            self._numerador * otra_f._denominador + self._denominador * otra_f._numerador,
            self._denominador * otra_f._denominador,
        )

    def __radd__(self, otra):
        return self.__add__(otra)

    def __sub__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return self + (-otra_f)

    def __rsub__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return otra_f - self

    def __mul__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return Fraccion(self._numerador * otra_f._numerador, self._denominador * otra_f._denominador)

    def __rmul__(self, otra):
        return self.__mul__(otra)

    def __truediv__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        if otra_f._numerador == 0:
            raise ZeroDivisionError("División por la fracción cero.")
        return Fraccion(self._numerador * otra_f._denominador, self._denominador * otra_f._numerador)

    def __rtruediv__(self, otra):
        try:
            otra_f = self.convertir(otra)
        except TypeError:
            return NotImplemented
        return otra_f / self


print("Aritmética exacta: (1/3 + 1/6) * 3 - 1/2 =", (Fraccion(1, 3) + Fraccion(1, 6)) * 3 - Fraccion(1, 2))

## 4. Proyecto integrador — Parte 2: `PolinomioRacional` en $\mathbb{Q}[x]$

Mediante composición, `PolinomioRacional` almacena una tupla inmutable de instancias de `Fraccion`. Gracias a ello, toda operación algebraica —incluidas derivación e integración definida mediante el segundo teorema fundamental del cálculo $\int_a^b P(x)\,dx = \Phi(b) - \Phi(a)$— produce resultados racionales exactos.

In [ ]:
class PolinomioRacional:
    variable = "x"

    def __init__(self, coeficientes):
        lista = [Fraccion.convertir(valor) for valor in coeficientes]
        if not lista:
            raise ValueError("Un polinomio debe tener al menos un coeficiente.")
        cero = Fraccion.cero()
        while len(lista) > 1 and lista[-1] == cero:
            lista.pop()
        self._coeficientes = tuple(lista)

    @classmethod
    def nulo(cls):
        return cls([Fraccion.cero()])

    @property
    def coeficientes(self):
        return self._coeficientes

    @property
    def grado(self):
        return len(self._coeficientes) - 1

    @property
    def es_nulo(self):
        return self.grado == 0 and self._coeficientes[0] == Fraccion.cero()

    def __len__(self):
        return len(self._coeficientes)

    def __repr__(self):
        return f"PolinomioRacional({list(self._coeficientes)})"

    def __str__(self):
        return " + ".join(
            f"({coeficiente})*{self.variable}^{exponente}"
            for exponente, coeficiente in enumerate(self._coeficientes)
        )

    def __eq__(self, otro):
        if not isinstance(otro, PolinomioRacional):
            return NotImplemented
        return self._coeficientes == otro._coeficientes

    def __neg__(self):
        return PolinomioRacional([-c for c in self._coeficientes])

    def __add__(self, otro):
        if not isinstance(otro, PolinomioRacional):
            return NotImplemented
        limite = max(len(self._coeficientes), len(otro._coeficientes))
        cero = Fraccion.cero()
        return PolinomioRacional(
            (self._coeficientes[i] if i < len(self._coeficientes) else cero)
            + (otro._coeficientes[i] if i < len(otro._coeficientes) else cero)
            for i in range(limite)
        )

    def __sub__(self, otro):
        if not isinstance(otro, PolinomioRacional):
            return NotImplemented
        return self + (-otro)

    def __mul__(self, otro):
        if not isinstance(otro, PolinomioRacional):
            return NotImplemented
        producto = [Fraccion.cero()] * (self.grado + otro.grado + 1)
        for i, a in enumerate(self._coeficientes):
            for j, b in enumerate(otro._coeficientes):
                producto[i + j] = producto[i + j] + a * b
        return PolinomioRacional(producto)

    def evaluar(self, valor):
        punto = Fraccion.convertir(valor)
        acumulado = Fraccion.cero()
        for coeficiente in reversed(self._coeficientes):
            acumulado = acumulado * punto + coeficiente
        return acumulado

    def derivar(self):
        if self.grado == 0:
            return PolinomioRacional.nulo()
        return PolinomioRacional(
            exponente * coeficiente
            for exponente, coeficiente in enumerate(self._coeficientes)
            if exponente > 0
        )

    def primitiva(self, constante=0):
        coeficientes_primitiva = [Fraccion.convertir(constante)]
        for exponente, coeficiente in enumerate(self._coeficientes):
            coeficientes_primitiva.append(coeficiente / (exponente + 1))
        return PolinomioRacional(coeficientes_primitiva)

    def integral_definida(self, a, b):
        antiderivada = self.primitiva()
        return antiderivada.evaluar(b) - antiderivada.evaluar(a)


p = PolinomioRacional([1, -1, 1])  # 1 - x + x^2
integral_exacta = p.integral_definida(0, 1)
print("P(x) =", p)
print("P'(x) =", p.derivar())
print("Primitiva de P(x):", p.primitiva())
print("Integral exacta en [0, 1]:", integral_exacta, "| repr:", repr(integral_exacta))
assert integral_exacta == Fraccion(5, 6)

## 5. Proyecto integrador — Parte 3: `FuncionRacional` en $\mathbb{Q}(x)$

Aplicamos un segundo nivel de composición: `FuncionRacional` contiene dos instancias de `PolinomioRacional` (numerador $P$ y denominador $Q \neq 0$), y calcula sumas, productos y derivadas mediante la regla del cociente:

$$\left(\frac{P}{Q}\right)' = \frac{P'Q - P Q'}{Q^2}.$$

In [ ]:
class FuncionRacional:
    def __init__(self, numerador, denominador=None):
        if not isinstance(numerador, PolinomioRacional):
            raise TypeError("El numerador debe ser un PolinomioRacional.")
        if denominador is None:
            denominador = PolinomioRacional([1])
        if not isinstance(denominador, PolinomioRacional):
            raise TypeError("El denominador debe ser un PolinomioRacional.")
        if denominador.es_nulo:
            raise ZeroDivisionError("El polinomio denominador no puede ser nulo.")
        self._numerador = numerador
        self._denominador = denominador

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    def evaluar(self, valor):
        valor_denominador = self._denominador.evaluar(valor)
        if valor_denominador == Fraccion.cero():
            raise ZeroDivisionError("La función racional tiene un polo en el punto evaluado.")
        return self._numerador.evaluar(valor) / valor_denominador

    def __add__(self, otra):
        if not isinstance(otra, FuncionRacional):
            return NotImplemented
        nuevo_numerador = self._numerador * otra._denominador + otra._numerador * self._denominador
        nuevo_denominador = self._denominador * otra._denominador
        return FuncionRacional(nuevo_numerador, nuevo_denominador)

    def __mul__(self, otra):
        if not isinstance(otra, FuncionRacional):
            return NotImplemented
        return FuncionRacional(self._numerador * otra._numerador, self._denominador * otra._denominador)

    def derivar(self):
        nuevo_numerador = (
            self._numerador.derivar() * self._denominador
            - self._numerador * self._denominador.derivar()
        )
        nuevo_denominador = self._denominador * self._denominador
        return FuncionRacional(nuevo_numerador, nuevo_denominador)

    def __repr__(self):
        return f"FuncionRacional({self._numerador!r}, {self._denominador!r})"

    def __str__(self):
        return f"({self._numerador}) / ({self._denominador})"


r1 = FuncionRacional(PolinomioRacional([1]), PolinomioRacional([1, 1]))   # 1 / (1 + x)
r2 = FuncionRacional(PolinomioRacional([0, 1]), PolinomioRacional([1, 1])) # x / (1 + x)
suma_r = r1 + r2
print("r1(1/2) + r2(1/2) =", suma_r.evaluar(Fraccion(1, 2)))
print("Derivada de 1 / (1 + x) en x = 1:", r1.derivar().evaluar(1))

## 6. Ejercicios

**Ejercicio 1 (`SegmentoPlano` por composición de dos `VectorPlano`).** Diseña la clase `SegmentoPlano` compuesta por dos instancias de `VectorPlano` (`inicio` y `fin`):
- **Entrada:** dos puntos `inicio` y `fin` de tipo `VectorPlano`.
- **Salida:** propiedades de solo lectura `inicio`, `fin`, `vector_director` (`VectorPlano(fin.x - inicio.x, fin.y - inicio.y)`), `longitud` (distancia entre `inicio` y `fin`), `punto_medio` y el método `trasladar(self, desplazamiento)` que devuelva un nuevo `SegmentoPlano`.

**Ejercicio 2 (integral definida exacta y simetría par en $\mathbb{Q}[x]$).** Considera el polinomio par
$$P(x) = \frac{1}{3} - 2x^2 + x^4.$$
- **Entrada:** la instancia correspondiente de `PolinomioRacional` y el intervalo simétrico $[-1, 1]$.
- **Salida:** verifica con aritmética racional exacta que `P.integral_definida(-1, 1)` coincide con `2 * P.integral_definida(0, 1)` y que ambos valen `Fraccion(-4, 15)`.

**Ejercicio 3 (derivada y evaluación exacta en `FuncionRacional`).** Define $R(x) = \frac{1 + x}{1 + x^2}$ mediante `FuncionRacional`:
- **Entrada:** los polinomios $P(x) = 1 + x$ y $Q(x) = 1 + x^2$, y el punto $x = \frac{1}{2}$.
- **Salida:** verifica exactamente que $R\left(\frac{1}{2}\right) = \frac{6}{5}$, que $R'\left(\frac{1}{2}\right) = -\frac{4}{25}$ y que $(R + R)\left(\frac{1}{2}\right) = \frac{12}{5}$.

## 7. Soluciones

In [ ]:
# Ejercicio 1
class SegmentoPlano:
    def __init__(self, inicio, fin):
        if not isinstance(inicio, VectorPlano) or not isinstance(fin, VectorPlano):
            raise TypeError("Los extremos del segmento deben ser instancias de VectorPlano.")
        self._inicio = inicio
        self._fin = fin

    @property
    def inicio(self):
        return self._inicio

    @property
    def fin(self):
        return self._fin

    @property
    def vector_director(self):
        return VectorPlano(self._fin.x - self._inicio.x, self._fin.y - self._inicio.y)

    @property
    def longitud(self):
        return self._inicio.distancia(self._fin)

    @property
    def punto_medio(self):
        return VectorPlano(
            (self._inicio.x + self._fin.x) / 2.0,
            (self._inicio.y + self._fin.y) / 2.0,
        )

    def trasladar(self, desplazamiento):
        return SegmentoPlano(self._inicio + desplazamiento, self._fin + desplazamiento)

    def __repr__(self):
        return f"SegmentoPlano({self._inicio!r}, {self._fin!r})"


seg = SegmentoPlano(VectorPlano(1, 2), VectorPlano(4, 6))
print("Segmento:", seg)
print("Vector director:", seg.vector_director, "| longitud:", seg.longitud, "| punto medio:", seg.punto_medio)
print("Trasladado por (-1, 1):", seg.trasladar(VectorPlano(-1, 1)))

In [ ]:
# Ejercicio 2
polinomio_par = PolinomioRacional([Fraccion(1, 3), 0, -2, 0, 1])
integral_total = polinomio_par.integral_definida(-1, 1)
doble_semintervalo = 2 * polinomio_par.integral_definida(0, 1)

assert integral_total == doble_semintervalo
assert integral_total == Fraccion(-4, 15)

print("P(x) =", polinomio_par)
print("Integral en [-1, 1]:", integral_total)
print("Doble en [0, 1]:    ", doble_semintervalo)

In [ ]:
# Ejercicio 3
r = FuncionRacional(PolinomioRacional([1, 1]), PolinomioRacional([1, 0, 1]))
x_prueba = Fraccion(1, 2)

valor_en_medio = r.evaluar(x_prueba)
derivada_en_medio = r.derivar().evaluar(x_prueba)
doble_en_medio = (r + r).evaluar(x_prueba)

assert valor_en_medio == Fraccion(6, 5)
assert derivada_en_medio == Fraccion(-4, 25)
assert doble_en_medio == Fraccion(12, 5)

print("R(1/2) =", valor_en_medio)
print("R'(1/2) =", derivada_en_medio)
print("(R + R)(1/2) =", doble_en_medio)

## 8. Resumen y mapa general de la serie

- Prefiere **composición** («tiene un») cuando un objeto se construye combinando otros objetos independientes (`Circunferencia`, `SegmentoPlano`, `PolinomioRacional`, `FuncionRacional`), y reserva la **herencia** («es un») para especializaciones genuinas.
- A lo largo de los ocho videos construimos paso a paso una biblioteca matemática coherente y extensible:

| Video | Tema central | Objeto matemático protagonista |
|---|---|---|
| 1 | Del procedimiento al objeto | Primera clase `Fraccion` |
| 2 | Clases, `__init__`, `self`, atributos de instancia y clase | `VectorPlano`, `VectorEspacio` y primer `Polinomio` |
| 3 | Métodos de instancia, `@classmethod` y `@staticmethod` | Constructores alternativos y algoritmo de Euclides |
| 4 | Encapsulamiento, `@property` e invariantes | `Fraccion`, `VectorPlano`, `Polinomio` e `IntervaloCerrado` protegidos |
| 5 | Métodos especiales (`__repr__`, `__str__`, `__eq__`, `__lt__`, `__add__`, `__len__`) | Operadores y orden en `Fraccion`, `Vector` y `Polinomio` |
| 6 | Herencia, `super()`, sobrescritura y `mro` | `PolinomioLineal`, `PolinomioCuadratico` y `EnteroRacional` |
| 7 | Polimorfismo y clases abstractas (`abc`) | `FuncionDiferenciable` con Newton-Raphson y trapecio |
| 8 | Composición frente a herencia y proyecto integrador | `PolinomioRacional` en $\mathbb{Q}[x]$ y `FuncionRacional` en $\mathbb{Q}(x)$ |